In [1]:
import ee
import geemap
import ipywidgets as widgets
from ipyleaflet import WidgetControl
import pandas as pd
import requests

# 1. Inicialización de Google Earth Engine
try:
    ee.Initialize(project='booming-list-511608-s2')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='booming-list-511608-s2')

# 2. Inicializar el mapa limpio
Map = geemap.Map(center=[22.58, -102.52], zoom=13, height='680px')
Map.layers = []

# Cargar la colección de Sentinel-2 Harmonized filtrada
sentinel_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                       .filterDate('2026-01-01', '2026-10-01')
                       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15)))

imagen_base = sentinel_collection.median()
Map.add_basemap('OpenStreetMap')

# 3. Widgets de control flotantes
index_selector = widgets.Dropdown(
    options=[
        ('NDVI', 'NDVI'),
        ('NDWI', 'NDWI'),
        ('MNDWI', 'MNDWI'),
        ('EVI', 'EVI'),
        ('NDSI', 'NDSI'),
        ('BSI', 'BSI'),
        ('SAVI', 'SAVI'),
        ('NDMI', 'NDMI')
    ],
    value='MNDWI',
    description='Estudio:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

# Widget numérico para definir el radio en metros (solo actúa si la figura es un marcador/punto)
input_radio = widgets.FloatText(
    value=50.0,
    description='Radio (m):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='140px')
)

btn_procesar = widgets.Button(
    description='Procesar Zona',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='130px')
)

btn_exportar = widgets.Button(
    description='Exportar Píxeles',
    button_style='info',
    icon='download',
    layout=widgets.Layout(width='135px')
)

control_leyenda_actual = None

# Estructura global para acumular todas las figuras procesadas y sus metadatos
historial_geometrias = []

# Fórmulas oficiales de índices centralizadas
def obtener_imagen_indice(estudio):
    if estudio == 'NDVI':
        img = imagen_base.normalizedDifference(['B8', 'B4']).rename('NDVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#8c510a', '#dfc27d', '#f5f5f5', '#80cdc1', '#01665e']}
        etiquetas = ['Muy Densa (0.8)', 'Densa (0.6)', 'Moderada (0.4)', 'Baja (0.2)', 'Suelo/Escaso (0.0)']
    elif estudio == 'NDWI':
        img = imagen_base.normalizedDifference(['B3', 'B8']).rename('NDWI')
        vis = {'min': -0.3, 'max': 0.5, 'palette': ['#d73027', '#fee08b', '#ffffbf', '#91bfdb', '#4575b4']}
        etiquetas = ['Agua Alta (0.5)', 'Agua Media (0.3)', 'Humedad (0.0)', 'Transicion (-0.1)', 'Suelo Seco (-0.3)']
    elif estudio == 'MNDWI':
        img = imagen_base.normalizedDifference(['B3', 'B11']).rename('MNDWI')
        vis = {'min': -0.5, 'max': 0.4, 'palette': ['#a50026', '#fdae61', '#ffffbf', '#abd9e9', '#08519c']}
        etiquetas = ['Agua Libre / Mar (0.4)', 'Agua Somera (0.2)', 'Transicion (0.0)', 'Suelo (-0.2)', 'Urbano / Roca (-0.5)']
    elif estudio == 'EVI':
        img = imagen_base.expression(
            '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
            {'NIR': imagen_base.select('B8'), 'RED': imagen_base.select('B4'), 'BLUE': imagen_base.select('B2')}
        ).rename('EVI')
        vis = {'min': 0.0, 'max': 1.0, 'palette': ['#f7fcb9', '#addd8e', '#78c679', '#31a354', '#006837']}
        etiquetas = ['Muy Alta (1.0)', 'Alta (0.75)', 'Media (0.5)', 'Baja (0.25)', 'Minima (0.0)']
    elif estudio == 'NDSI':
        img = imagen_base.normalizedDifference(['B3', 'B11']).rename('NDSI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#000000', '#c6dbef', '#9ecae1', '#4292c6', '#ffffff']}
        etiquetas = ['Nieve Alta (0.8)', 'Media (0.6)', 'Baja (0.4)', 'Minima (0.2)', 'Sin Nieve (0.0)']
    elif estudio == 'BSI':
        img = imagen_base.expression(
            '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
            {'SWIR': imagen_base.select('B11'), 'RED': imagen_base.select('B4'), 'NIR': imagen_base.select('B8'), 'BLUE': imagen_base.select('B2')}
        ).rename('BSI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#3288bd', '#abdda4', '#ffffbf', '#fdae61', '#d53e4f']}
        etiquetas = ['Suelo Denso (0.4)', 'Alto (0.2)', 'Neutro (0.0)', 'Bajo (-0.2)', 'Vegetacion/Agua (-0.4)']
    elif estudio == 'SAVI':
        img = imagen_base.expression(
            '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
            {'NIR': imagen_base.select('B8'), 'RED': imagen_base.select('B4')}
        ).rename('SAVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#225ea8', '#0c2c84']}
        etiquetas = ['Alta (0.8)', 'Media-Alta (0.6)', 'Media (0.4)', 'Baja (0.2)', 'Minima (0.0)']
    elif estudio == 'NDMI':
        img = imagen_base.normalizedDifference(['B8', 'B11']).rename('NDMI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#4575b4']}
        etiquetas = ['Alta Humedad (0.4)', 'Media-Alta (0.2)', 'Moderada (0.0)', 'Baja (-0.2)', 'Muy Seco (-0.4)']
    return img, vis, estudio, etiquetas

# 4. Función de análisis espacial para la última figura creada (acumulándola en el historial)
def ejecutar_analisis_individual(b):
    global control_leyenda_actual, historial_geometrias
    estudio = index_selector.value
    radio_metros = input_radio.value
    
    geometria_base = None
    nombre_figura = "Desconocido"
    
    try:
        if hasattr(Map, 'draw_features') and len(Map.draw_features) > 0:
            idx = len(Map.draw_features)
            ultimo_dibujo = Map.draw_features[-1]
            
            if hasattr(ultimo_dibujo, 'geometry'):
                g = ultimo_dibujo.geometry()
            elif isinstance(ultimo_dibujo, dict):
                g = ee.Geometry(ultimo_dibujo.get('geometry', ultimo_dibujo))
            else:
                g = ee.Feature(ultimo_dibujo).geometry()
            
            tipo_g = g.type().getInfo()
            
            if tipo_g == 'Point':
                geometria_base = g.buffer(radio_metros)
                nombre_figura = f"Radio_{int(radio_metros)}m_{idx}"
            else:
                geometria_base = g
                nombre_figura = f"Figura_{idx}_{tipo_g}"
                
    except Exception as e:
        print(f"Error al capturar la figura: {e}")
        
    if geometria_base is None:
        print("Atención: Dibuja primero una geometría o coloca un marcador antes de procesar.")
        return

    # Guardamos la figura procesada en la lista global
    historial_geometrias.append({
        'geometry': geometria_base,
        'nombre': f"{nombre_figura}_{estudio}"
    })

    layer_img, vis, titulo_leyenda, etiquetas = obtener_imagen_indice(estudio)
    imagen_recortada = layer_img.clip(geometria_base)
    
    Map.addLayer(imagen_recortada, vis, f'{nombre_figura}: {estudio}')
    
    if control_leyenda_actual is not None:
        try:
            Map.remove_control(control_leyenda_actual)
        except Exception:
            pass

    html_leyenda = f"""
    <div style="background: white; padding: 10px; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2); font-family: sans-serif; font-size: 10px; width: 190px;">
        <b>{titulo_leyenda}</b><br>
        <div style="background: linear-gradient(to top, {vis['palette'][0]}, {vis['palette'][1]}, {vis['palette'][2]}, {vis['palette'][3]}, {vis['palette'][4]}); width: 18px; height: 120px; float: left; margin-right: 8px; margin-top: 5px;"></div>
        <div style="height: 120px; display: inline-block; line-height: 24px;">
            <span>{etiquetas[0]}<br></span>
            <span>{etiquetas[1]}<br></span>
            <span>{etiquetas[2]}<br></span>
            <span>{etiquetas[3]}<br></span>
            <span>{etiquetas[4]}</span>
        </div>
    </div>
    """
    legend_widget = widgets.HTML(value=html_leyenda)
    control_leyenda_actual = WidgetControl(widget=legend_widget, position='bottomleft')
    Map.add_control(control_leyenda_actual)
    
    try:
        Map.centerObject(geometria_base, zoom=14)
    except Exception:
        pass
        
    print(f'Proceso completado para {nombre_figura} con el índice: {estudio}.')

btn_procesar.on_click(lambda b: ejecutar_analisis_individual(b))

# 5. Función para exportar los píxeles de TODAS las figuras procesadas a un solo CSV de forma segura
def exportar_pixeles_zona(b):
    if len(historial_geometrias) == 0:
        print("Atención: Primero procesa al menos una zona en el mapa.")
        return

    estudio = index_selector.value
    print(f"Extrayendo píxeles de todas las figuras procesadas...")
    
    layer_img, _, _, _ = obtener_imagen_indice(estudio)
    
    try:
        lista_datos = []
        contador_global = 1

        for item in historial_geometrias:
            geom_actual = item['geometry']
            nombre_origen = item['nombre']
            
            # Se aplica numPixels=4999 de manera independiente por figura para evitar el límite de GEE
            muestras = layer_img.sample(
                region=geom_actual,
                scale=10,  # Resolución nativa de 10m de Sentinel-2
                geometries=True,
                numPixels=4999
            ).getInfo()['features']

            for feat in muestras:
                props = feat['properties']
                coords = feat['geometry']['coordinates']
                lista_datos.append({
                    'Pixel_ID': f'PXL_{contador_global:04d}',
                    'Origen_Figura': nombre_origen,
                    'Longitud': coords[0],
                    'Latitud': coords[1],
                    estudio: props.get(estudio, None)
                })
                contador_global += 1

        if len(lista_datos) == 0:
            print("No se encontraron píxeles en las zonas procesadas.")
            return

        df = pd.DataFrame(lista_datos)
        csv_name = f'analisis_{estudio.lower()}_multizona_etiquetado.csv'
        df.to_csv(csv_name, index=False)
        
        print(f"¡Exportación exitosa! Se procesaron {len(df)} píxeles de todas las figuras.")
        print(f" Archivo guardado: '{csv_name}'.")

    except Exception as ex:
        print(f"Error en la extracción: {ex}")

btn_exportar.on_click(exportar_pixeles_zona)

# 6. Panel de controles
panel_controles = widgets.HBox([index_selector, input_radio, btn_procesar, btn_exportar])
control_menu = WidgetControl(widget=panel_controles, position='bottomright')
Map.add_control(control_menu)

Map

Map(center=[22.58, -102.52], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…